In [1]:
import random
from typing import Callable, Optional

In [2]:
class Individuo:

    def __init__(self, num_genes: int, cromosoma: Optional[list[int]] = None):
        self.num_genes = num_genes
        self.cromosoma = cromosoma if cromosoma else [random.randint(0, 1) for _ in range(num_genes)]
        self.fitness: Optional[float] = None

    def evaluar(self, funcion_aptitud: Callable[[list[int]], float]) -> float:
        self.fitness = funcion_aptitud(self.cromosoma)
        return self.fitness

    def mutar(self, tasa_mutacion: float = 0.01):
        for i in range(self.num_genes):
            if random.random() < tasa_mutacion:
                self.cromosoma[i] = 1 - self.cromosoma[i]

    def fenotipo(self) -> int:
        return int(''.join(str(g) for g in self.cromosoma), 2)

    def __repr__(self) -> str:
        return f"Individuo(cromosoma={''.join(str(g) for g in self.cromosoma)}, fitness={self.fitness})"

In [3]:
class Poblacion:

    def __init__(self, tamano: int, num_genes: int, tasa_mutacion: float = 0.01, tasa_cruzamiento: float = 0.7):
        self.tamano = tamano
        self.num_genes = num_genes
        self.tasa_mutacion = tasa_mutacion
        self.tasa_cruzamiento = tasa_cruzamiento
        self.generacion = 0
        self.individuos = [Individuo(num_genes) for _ in range(tamano)]

    def evaluar(self, funcion_aptitud: Callable[[list[int]], float]):
        for individuo in self.individuos:
            individuo.evaluar(funcion_aptitud)

    def mejor_individuo(self) -> Individuo:
        return max(self.individuos, key=lambda ind: ind.fitness or 0)

    def seleccion_torneo(self, tamano_torneo: int = 3) -> Individuo:
        competidores = random.sample(self.individuos, tamano_torneo)
        return max(competidores, key=lambda ind: ind.fitness or 0)

    def cruzamiento(self, padre1: Individuo, padre2: Individuo) -> tuple[Individuo, Individuo]:
        if random.random() > self.tasa_cruzamiento:
            return (Individuo(self.num_genes, padre1.cromosoma[:]),
                    Individuo(self.num_genes, padre2.cromosoma[:]))

        punto = random.randint(1, self.num_genes - 1)
        hijo1 = padre1.cromosoma[:punto] + padre2.cromosoma[punto:]
        hijo2 = padre2.cromosoma[:punto] + padre1.cromosoma[punto:]
        return (Individuo(self.num_genes, hijo1), Individuo(self.num_genes, hijo2))

    def evolucionar(self, funcion_aptitud: Callable[[list[int]], float]):
        nueva = [Individuo(self.num_genes, self.mejor_individuo().cromosoma[:])]

        while len(nueva) < self.tamano:
            p1, p2 = self.seleccion_torneo(), self.seleccion_torneo()
            h1, h2 = self.cruzamiento(p1, p2)
            h1.mutar(self.tasa_mutacion)
            h2.mutar(self.tasa_mutacion)
            nueva.append(h1)
            if len(nueva) < self.tamano:
                nueva.append(h2)

        self.individuos = nueva
        self.evaluar(funcion_aptitud)
        self.generacion += 1

    def __repr__(self) -> str:
        return f"Poblacion(gen={self.generacion}, tamano={self.tamano}, mejor_fitness={self.mejor_individuo().fitness})"

In [4]:
def fitness_onemax(cromosoma: list[int]) -> float:
    return sum(cromosoma)


random.seed(42)
poblacion = Poblacion(tamano=30, num_genes=20, tasa_mutacion=0.02, tasa_cruzamiento=0.8)
poblacion.evaluar(fitness_onemax)

print("=== Evolución del Algoritmo Genético ===")
print(f"Generación 0 -> Mejor fitness: {poblacion.mejor_individuo().fitness}")

for gen in range(1, 51):
    poblacion.evolucionar(fitness_onemax)
    mejor = poblacion.mejor_individuo()

    if gen % 10 == 0 or mejor.fitness == 20:
        print(f"Generación {gen} -> Mejor fitness: {mejor.fitness}")

    if mejor.fitness == 20:
        print(f"\nSolución óptima encontrada en la generación {gen}.")
        break

print(f"\n=== Resultado Final ===")
print(f"{poblacion}")

=== Evolución del Algoritmo Genético ===
Generación 0 -> Mejor fitness: 14
Generación 10 -> Mejor fitness: 19
Generación 13 -> Mejor fitness: 20

Solución óptima encontrada en la generación 13.

=== Resultado Final ===
Poblacion(gen=13, tamano=30, mejor_fitness=20)


In [5]:
print("Primeros 5 individuos de la población final:")
print()
for i, ind in enumerate(poblacion.individuos[:5]):
    print(f"  #{i+1}: {ind}")
    print(f"       Fenotipo (valor decimal): {ind.fenotipo()}")

fitness_values = [ind.fitness for ind in poblacion.individuos if ind.fitness is not None]
print(f"\nEstadísticas de la población final:")
print(f"  Fitness promedio: {sum(fitness_values) / len(fitness_values):.2f}")
print(f"  Fitness mínimo:   {min(fitness_values)}")
print(f"  Fitness máximo:   {max(fitness_values)}")

Primeros 5 individuos de la población final:

  #1: Individuo(cromosoma=11111111111011111111, fitness=19)
       Fenotipo (valor decimal): 1048319
  #2: Individuo(cromosoma=11111111101011111111, fitness=18)
       Fenotipo (valor decimal): 1047295
  #3: Individuo(cromosoma=11111111111011111111, fitness=19)
       Fenotipo (valor decimal): 1048319
  #4: Individuo(cromosoma=11111111111011111111, fitness=19)
       Fenotipo (valor decimal): 1048319
  #5: Individuo(cromosoma=01111111111011111111, fitness=18)
       Fenotipo (valor decimal): 524031

Estadísticas de la población final:
  Fitness promedio: 18.23
  Fitness mínimo:   16
  Fitness máximo:   20
